# 🎬 Дослідницький аналіз даних (EDA) - IMDb Movie Reviews

**Дисципліна:** Глибинне навчання (ІПЗ-4)  
**Тема:** Movie Review Sentiment Analysis (IMDb 50k Dataset)  
**Виконавець:** Денис (Data & Deployment Lead)  
**Мета:** Провести всебічний аналіз корпусу текстів відгуків, оцінити розподіл класів, довжину повідомлень, словниковий запас, побудувати n-грами та виявити ключові патерни для подальшого препроцесингу й моделювання нейромережами.

## 1. Імпорт бібліотек та налаштування середовища

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

# Візуальні стилі
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (10, 6)

print("Бібліотеки успішно імпортовано!")

## 2. Завантаження датасету
> **Примітка:** Датасет завантажується з локальної папки `../data/IMDB Dataset.csv` або за прямим посиланням.

In [ ]:
DATA_PATH = os.path.join("..", "data", "IMDB Dataset.csv")

# Перевірка наявності локального файлу або завантаження з резервного джерела
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Датасет завантажено з: {DATA_PATH}")
else:
    print("Локальний файл не знайдено, завантажуємо безпосередньо з хмари (IMDb 50k)...")
    url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
    df = pd.read_csv(url)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Датасет завантажено та збережено в: {DATA_PATH}")

print(f"Форма датасету: {df.shape[0]} рядків, {df.shape[1]} стовпців")
df.head()

## 3. Базова інформація та перевірка цілісності даних

In [ ]:
# Загальна інформація про типи даних
df.info()

# Перевірка на пропущені значення (Missing Values)
missing_values = df.isnull().sum()
print("\nПропущені значення за колонками:")
print(missing_values)

# Перевірка дублікатів
duplicates_count = df.duplicated().sum()
print(f"\nКількість повних дублікатів рядків: {duplicates_count}")
if duplicates_count > 0:
    print("Рекомендація: дублікати відгуків слід видалити на етапі препроцесингу.")

## 4. Аналіз балансу цільової змінної (`sentiment`)

In [ ]:
sentiment_counts = df['sentiment'].value_counts()
print("Розподіл класів:")
print(sentiment_counts)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette=['#4CAF50', '#F44336'], ax=ax[0])
ax[0].set_title("Кількість відгуків за класами", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Клас (Sentiment)")
ax[0].set_ylabel("Кількість")
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                   textcoords='offset points')

# Pie chart
ax[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', 
         startangle=90, colors=['#4CAF50', '#F44336'], explode=[0.03, 0.03])
ax[1].set_title("Співвідношення позитивних і негативних відгуків", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Висновки: Датасет ідеально збалансований (50% positive, 50% negative), що дозволяє використовувати Accuracy та F1-Score без корекції ваг класів.")

## 5. Аналіз характеристик тексту (довжина, слова, речення)

In [ ]:
# Розрахунок довжини тексту в символах та словах
df['char_length'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

print("Статистичні характеристики довжини відгуків у словах:")
print(df.groupby('sentiment')['word_count'].describe())

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Гістограма розподілу кількості слів
sns.histplot(data=df, x='word_count', hue='sentiment', bins=60, kde=True, 
             palette=['#4CAF50', '#F44336'], element='step', ax=ax[0])
ax[0].set_title("Розподіл кількості слів у відгуках", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Кількість слів")
ax[0].set_ylabel("Густина / Кількість")
ax[0].set_xlim(0, 1200)

# Boxplot
sns.boxplot(data=df, x='sentiment', y='word_count', palette=['#4CAF50', '#F44336'], ax=ax[1])
ax[1].set_title("Boxplot кількості слів за класами", fontsize=14, fontweight='bold')
ax[1].set_xlabel("Клас")
ax[1].set_ylabel("Кількість слів")
ax[1].set_ylim(0, 1200)

plt.tight_layout()
plt.show()

## 6. Виявлення HTML-тегів та шуму в текстах
Оскільки відгуки зібрані з веб-сторінок IMDb, вони часто містять специфічні HTML-артефакти (наприклад `<br /><br />`).

In [ ]:
# Перевірка наявності HTML тегів
html_tags_pattern = r'<.*?>'
reviews_with_html = df['review'].apply(lambda x: bool(re.search(html_tags_pattern, x)))
print(f"Відсоток відгуків з HTML-тегами: {reviews_with_html.mean() * 100:.2f}%")

# Приклад типового відгуку з артефактами
sample_review_html = df[reviews_with_html]['review'].iloc[0]
print("\nПриклад тексту з HTML-тегами:")
print(sample_review_html[:300] + "...")

## 7. Хмари слів (WordCloud) для позитивних та негативних відгуків

In [ ]:
# Завантаження стоп-слів NLTK
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
# Додаємо частотні контекстні слова, які не несуть сентименту
custom_stops = stop_words.union({'br', 'movie', 'film', 'one', 'make', 'see', 'watch'})

pos_text = " ".join(df[df['sentiment'] == 'positive']['review'].sample(2500, random_state=42))
neg_text = " ".join(df[df['sentiment'] == 'negative']['review'].sample(2500, random_state=42))

wc_pos = WordCloud(width=800, height=400, background_color='white', stopwords=custom_stops, colormap='Greens').generate(pos_text)
wc_neg = WordCloud(width=800, height=400, background_color='white', stopwords=custom_stops, colormap='Reds').generate(neg_text)

fig, ax = plt.subplots(1, 2, figsize=(18, 7))
ax[0].imshow(wc_pos, interpolation='bilinear')
ax[0].set_title("Найчастіші слова у позитивних відгуках", fontsize=16, fontweight='bold', color='darkgreen')
ax[0].axis('off')

ax[1].imshow(wc_neg, interpolation='bilinear')
ax[1].set_title("Найчастіші слова у негативних відгуках", fontsize=16, fontweight='bold', color='darkred')
ax[1].axis('off')

plt.tight_layout()
plt.show()

## 8. Аналіз біграм (N-grams Analysis)

In [ ]:
def extract_bigrams(text_series, top_n=10):
    clean_pattern = re.compile(r'[^a-zA-Z\s]')
    bigram_counter = Counter()
    for text in text_series:
        words = [w for w in clean_pattern.sub('', text.lower()).split() if w not in custom_stops and len(w) > 2]
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
        bigram_counter.update(bigrams)
    return pd.DataFrame(bigram_counter.most_common(top_n), columns=['bigram', 'count'])

pos_bigrams = extract_bigrams(df[df['sentiment'] == 'positive']['review'].sample(3000, random_state=42))
neg_bigrams = extract_bigrams(df[df['sentiment'] == 'negative']['review'].sample(3000, random_state=42))

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=pos_bigrams, y='bigram', x='count', palette='Greens_r', ax=ax[0])
ax[0].set_title("Топ-10 біграм: Позитивні відгуки", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Частота")

sns.barplot(data=neg_bigrams, y='bigram', x='count', palette='Reds_r', ax=ax[1])
ax[1].set_title("Топ-10 біграм: Негативні відгуки", fontsize=14, fontweight='bold')
ax[1].set_xlabel("Частота")

plt.tight_layout()
plt.show()

## 9. Підсумкові висновки для етапу моделювання

1. **Баланс класів:** Датасет ідеально симетричний (50/50), модифікація вагів у `loss` функції або resampling не потрібні.
2. **Очищення тексту:** Майже всі відгуки містять HTML-теги (`<br />`), що вимагає обов'язкової фільтрації через `BeautifulSoup` або regex перед токенізацією.
3. **Довжина послідовностей (Sequence Length):** Більшість відгуків містять до 250-300 слів. Довгий хвіст розподілу досягає понад 1000 слів. Для моделі LSTM доцільно встановити `max_length = 200...250` для збереження 90%+ інформації при оптимальній швидкості навчання.
4. **Словниковий запас (Vocabulary Size):** Рекомендований розмір словника для Embedding шару: 10 000 - 20 000 найчастіших токенів.